# 1. Encoder–Decoder 구조

- Encoder–Decoder 구조는 어떤 형태의 입력 시퀀스를 받아 **의미를 해석**한 뒤, 새로운 **출력 시퀀스를 생성**해야 하는 거의 모든 AI 문제를 해결하는 딥러닝 모델 구조다.
- 이 구조는 Encoder와 Decoder 두개의 딥러닝 모델을 연결한 구조로 **입력 데이터를 하나의 표현으로 압축한 뒤, 이를 다시 출력 데이터로 변환하는 방식**으로 동작한다.

- **Encoder Network**
  - 입력 데이터를 해석(이해)하는 역할을 수행한다.
  - 입력 시퀀스에 담긴 의미적 정보를 하나의 고정된 벡터 형태로 요약한다.

- **Decoder Network**
  - Encoder가 생성한 요약 정보를 바탕으로 최종 출력을 생성한다.
  - 즉, Encoder의 “이해 결과”를 이용해 새로운 시퀀스를 만들어낸다.

## 1.1 Seq2Seq (Sequence-to-Sequence)

Seq2Seq 모델은 **Encoder–Decoder 구조를 RNN(Recurrent Neural Network) 계열에 적용한 대표적인 시퀀스 변환 모델**이다.  
입력과 출력이 모두 “시퀀스(sequence)” 형태라는 점에서 *Sequence-to-Sequence*라는 이름이 붙었다.

### 1.1.1 Encoder의 역할: 입력 시퀀스 이해 및 Context Vector 생성

Encoder는 입력으로 들어온 **전체 시퀀스**(sequence)를 순차적으로 처리한 뒤,  그 의미를 **하나의 고정 길이 벡터**(Vector)로 압축하여 출력한다.  
이 벡터를 **Context Vector**(컨텍스트 벡터)라고 한다.
- **Context Vector란?**  
  - 입력 시퀀스 전체의 의미, 문맥, 핵심 정보를 요약해 담고 있는 벡터 표현이다.
  - **기계 번역**(Machine Translation)의 경우  
    - 번역할 원문 문장에서 **번역 결과를 생성하는 데 필요한 핵심 의미 정보**(feature)
  - **챗봇**(Chatbot)의 경우  
    - 사용자가 입력한 질문에서 **적절한 답변을 생성하는 데 필요한 의미 정보**(feature)

### 1.1.2 Decoder의 역할: Context Vector를 바탕으로 출력 시퀀스 생성

Decoder는 Encoder가 출력한 **Context Vector를 입력으로 받아**, 이를 바탕으로 **목표 출력 시퀀스**를 한 토큰(token)씩 순차적으로 생성한다.

- **기계 번역**(Machine Translation)의 경우  
  - 입력 문장의 의미를 반영한 **번역 문장**을 생성한다.
- **챗봇**(Chatbot)  
  - 질문에 대한 **자연스러운 답변 문장**을 생성한다.

Decoder는 매 시점(time step)마다
  - 이전에 생성한 단어
  - 그리고 Context Vector에 담긴 입력 문맥
을 함께 고려하여 다음 단어를 예측한다.


![seq2seq](figures/seq2seq.png)

# 2. Seq2Seq 를 이용한 Chatbot 모델 구현
- Encoder를 이용해 질문의 특성을 추출하고 Decoder를 이용해 답변을 생성한다.

# 3. Chatbot Dataset

- https://github.com/songys/Chatbot_data
- columns
    - Q: 질문
    - A: 답
    - label: 일상다반사 0, 이별(부정) 1, 사랑(긍정) 2
- **Download**

![dataset](figures/chatbot.png)

# 4. Chatbot Dataset Loading 및 확인

## 4.1 데이터셋 다운로드 및 확인

In [2]:
url = "https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv"

In [3]:
import requests
import os
os.makedirs("data/chatbot", exist_ok=True)
res  = requests.get(url)
if res.status_code == 200:
    with open("data/chatbot/chatbot_data.csv", "wt", encoding="utf-8") as fw:
        fw.write(res.text)
else:
    print("다운로드 실패 - ", res.status_code)

In [4]:
import pandas as pd
df = pd.read_csv("data/chatbot/chatbot_data.csv", encoding="utf-8")
df.shape

(11823, 3)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11823 entries, 0 to 11822
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Q       11823 non-null  str  
 1   A       11823 non-null  str  
 2   label   11823 non-null  int64
dtypes: int64(1), str(2)
memory usage: 277.2 KB


In [6]:
# 라벨 대화분류 컬럼 -> 제거
df.drop(columns=["label"], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11823 entries, 0 to 11822
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Q       11823 non-null  str  
 1   A       11823 non-null  str  
dtypes: str(2)
memory usage: 184.9 KB


In [7]:
df.head()

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.


In [8]:
df.tail(10)

,Q,A
11813,회사에 좋아하는 남자가 생겼어 어떡하지?,사랑하기 힘든 관계인가봐요.
11814,회사에서 어떤 사람이랑 자꾸 눈 마추쳐.,눈 마주치는 게 우연인지 잘 살펴 보세요.
11815,회식 중이라고 하는데 연락이 안돼.,정신 없이 바쁠지도 몰라요. 조금만 더 기다려보고 물어보는게 좋을 것 같아요.
11816,회식하는데 나만 챙겨줘. 썸임?,호감이 있을 수도 있어요. 그렇지만 조금 더 상황을 지켜보세요.
11817,후회 없이 사랑하고 싶어,진심으로 다가가 보세요.
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.
11822,힘들어서 결혼할까봐,도피성 결혼은 하지 않길 바라요.


In [9]:
# 결측치 확인
df.isnull().sum()

Q    0
A    0
dtype: int64

# 5. Dataset, DataLoader 정의

## 5.1 Tokenization

### 5.1.1 Subword방식

In [10]:
question_texts = df["Q"]
answer_texts = df["A"]
# 토큰화를 위해서 질문 +답변을 합쳐서 한나의 문서로 만든다.
all_texts = list(question_texts + " " + answer_texts) # 시리즈 + 시리즈 : 원소별 연산, 어휘사전을 만들기 위헤


In [11]:
all_texts[:5]

['12시 땡! 하루가 또 가네요.',
 '1지망 학교 떨어졌어 위로해 드립니다.',
 '3박4일 놀러가고 싶다 여행은 언제나 좋죠.',
 '3박4일 정도 놀러가고 싶다 여행은 언제나 좋죠.',
 'PPL 심하네 눈살이 찌푸려지죠.']

In [12]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=30000,
    min_frequency=2,
    continuing_subword_prefix="##",
    special_tokens=["<unk>", "<pad>", "<sos>"],
)
# 토큰나이저 학습
tokenizer.train_from_iterator(all_texts, trainer=trainer)



In [13]:
print(tokenizer.get_vocab_size())

15579


In [14]:
e = tokenizer.encode("3박4일 정도 놀러가고 싶다")
e.tokens
e.ids

[15118, 2892, 5235, 2331]

### 5.1.2 Tokenizer 저장

In [15]:
os.makedirs("saved_model/chatbot_seq2seq", exist_ok=True)
tokenizer.save("saved_model/chatbot_seq2seq/chatbot_bpe.json")

## 5.2 Dataset, DataLoader 정의


### 5.2.1 Dataset 정의 및 생성
- 모든 문장의 토큰 수는 동일하게 맞춰준다.
    - DataLoader는 batch 를 구성할 때 batch에 포함되는 데이터들의 shape이 같아야 한다. 그래야 하나의 batch로 묶을 수 있다.
    - 문장의 최대 길이를 정해주고 **최대 길이보다 짧은 문장은 `<PAD>` 토큰을 추가**하고 **최대길이보다 긴 문장은 최대 길이에 맞춰 짤라준다.**

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [17]:
class ChatbotDataset(Dataset):
    def __init__(self, question_texts, answer_text, max_length, tokenizer):
        """
        Args:
            question_texts: list[str] - 질문 text들
            answer_text: list[str]
            max_length: int - 개별 질문, 답변의 토큰 수
            tokenizer: Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        # [Tensor([토큰 id ...])]
        self.question_texts = [self.__process_sequence(q) for q in question_texts]
        self.answer_texts = [self.__process_sequence(a) for a in answer_texts]
    def __process_sequence(self, text:str):
        """
        한 문자을 받아서 맥스렝스 크기의 토근 텐서로 반환
        """
        # 토큰화
        encoding = self.tokenizer.encode(text)
        # 맥스렝쓰 크기에 맞추기(모자라면 <pad> 추가, 먼치면 잘라내고)
        tokens = self.__pad_token_sequence(encoding.ids)
        return torch.tensor(tokens, dtype=torch.int64)
    def __pad_token_sequence(self, token_sequence):
        """
        token sequence : [[토큰 ID, ...], [...]]를 맥스렝쓰에 크기를 맞춰준다
        """
        pad_token = self.tokenizer.token_to_id("<pad>")
        seq_length = len(token_sequence)
        if seq_length >= self.max_length:
            result = token_sequence[:self.max_length]
        else:
            result = token_sequence + [pad_token] * (self.max_length - seq_length)
        return result
    def __len__(self):
        return len(self.question_texts)
    def __getitem__(self, index):
        """
        index의 X(question), y(answer)을 튜플로 반화
        """
        q = self.question_texts[index]
        a = self.answer_texts[index]
        return q, a

In [18]:
# max_length - QA중에서 가장 긴 토큰으로 맞춘다.
max([ len(tokenizer.encode(sent).ids) for sent in question_texts])

18

In [19]:
max([ len(tokenizer.encode(sent).ids) for sent in answer_texts])

26

In [20]:
max_length = 26
dataset = ChatbotDataset(
    question_texts,
    answer_texts,
    max_length,
    tokenizer,
)

In [25]:
question_texts[1200], answer_texts[1200]

('대리운전 불렀어', '잘했어요.')

In [21]:
dataset[1200]

(tensor([5318, 1340, 1310,  591, 7706,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1]),
 tensor([4538,    8,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1]))

In [22]:
len(dataset)

11823

### 5.2.2 Trainset / Testset 나누기
train : test = 0.95 : 0.05

In [23]:
train_size = int(len(dataset)* 0.95)
test_size = len(dataset) - train_size
train_size, test_size

(11231, 592)

In [28]:
trainset, testset = random_split(dataset, [train_size, test_size])
len(trainset), len(testset)

(11231, 592)

### 5.2.3 DataLoader 생성

In [30]:
train_loader = DataLoader(trainset, batch_size=100, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=100)

In [31]:
len(train_loader), len(test_loader)

(112, 6)

# 6. 모델 정의

## 6.1 Encoder
Encoder는 하나의 Vector를 생성하며 그 Vector는 **입력 문장의 의미**를 N 차원 공간 저장하고 있다. 이 Vector를 **Context Vector** 라고 한다.    
![encoder](figures/seq2seq_encoder.png)

## 6.2 Decoder
- Encoder의 출력(context vector)를 받아서 답변 결과 sequence를 출력한다.
- Decoder는 매 time step의 입력으로 **이전 time step에서 예상한 단어와 hidden state값이** 입력된다.
- Decoder의 처리결과 hidden state를 Estimator(Linear+Softmax)로 입력하여 **입력 단어에 대한 번역 단어가 출력된다.** (이 출력단어가 다음 step의 입력이 된다.)
    - Decoder의 첫 time step 입력은 문장의 시작을 의미하는 <SOS>(start of string) 토큰이고 hidden state는 context vector(encoder 마지막 hidden state) 이다.

![decoder](figures/seq2seq_decoder.png)

## 6.3 Seq2Seq 모델

- Encoder - Decoder 를 Layer로 가지며 Encoder로 질문의 feature를 추출하고 Decoder로 답변을 생성한다.

### 6.3.1 Teacher Forcing
- **Teacher forcing** 기법은, RNN계열 모델이 다음 단어를 예측할 때, 이전 timestep에서 예측된 단어를 입력으로 사용하는 대신 **실제 정답 단어(ground truth) 단어를** 입력으로 사용하는 방법이다.
    - 모델은 이전 시점의 출력 단어를 다음 시점의 입력으로 사용한다. 그러나 모델이 학습할 때 초반에는 정답과 많이 다른 단어가 생성되어 엉뚱한 입력이 들어가 학습이 빠르게 되지 않는 문제가 있다.
- **장점**
    - **수렴 속도 증가**: 정답 단어를 사용하기 때문에 모델이 더 빨리 학습할 수있다.
    - **안정적인 학습**: 초기 학습 단계에서 모델의 예측이 불안정할 때, 잘못된 예측으로 인한 오류가 다음 단계로 전파되는 것을 막아줍니다.
- **단점**
    - **노출 편향(Exposure Bias) 문제:** 실제 예측 시에는 정답을 제공할 수 없으므로 모델은 전단계의 출력값을 기반으로 예측해 나가야 한다. 학습 과정과 추론과정의 이러한 차이 때문에 모델의 성능이 떨어질 수있다.
        - 이런 문제를 해결하기 학습 할 때 **Teacher forcing을 random하게 적용하여 학습시킨다.**

![seq2seq](figures/seq2seq.png)

# 7. 학습

## 7.1 모델생성

## 7.2 loss함수, optimizer

## 7.3 train/evaluation 함수 정의

### 7.3.1 train 함수정의

### 7.3.2 Test 함수

### 7.3.3 Training

# 8. 결과확인

- Sampler:
    -  DataLoader가 Datatset의 값들을 읽어서 batch를 만들때 index 순서를 정해주는 객체.
    -  DataLoader의 기본 sampler는 SequentialSampler 이다. shuffle=True 일경우 RandomSampler: 랜덤한 순서로 제공.

# 9. 학습모델을 이용한 대화